In [1]:
import numpy as np
import pandas as pd

raw_path_19   = r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\ACS Data\acs_2019_suffolk_tracts.csv'
clean_path_19 = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed\acs_2019_tracts_clean.csv'


acs19 = pd.read_csv(raw_path_19, dtype={'state': str, 'county': str, 'tract': str})
print(acs19.shape)

(204, 29)


In [ ]:
rename_map = {
    'NAME':        'tract_name',          # tract name
    'B19013_001E': 'median_hh_income',    # median household income ($)
    'B17001_001E': 'pov_universe',        # people whose poverty status is known
    'B17001_002E': 'pov_below',           # people below the poverty line
    'B08201_001E': 'households',          # total households
    'B08201_002E': 'hh_no_vehicle',       # households with no vehicle
    'B03002_001E': 'pop_total',           # total population
    'B03002_003E': 'pop_nh_white',        # non-Hispanic White
    'B03002_004E': 'pop_nh_black',        # non-Hispanic Black
    'B03002_006E': 'pop_nh_asian',        # non-Hispanic Asian
    'B03002_012E': 'pop_hispanic',        # Hispanic or Latino
    'B08301_001E': 'workers',             # workers 16+
    'B08301_010E': 'workers_transit',     # workers commuting by public transit
    'B08303_001E': 'commuters',           # workers who travel to work
    'B08303_011E': 'commute_45_59min',    # commute 45-59 minutes
    'B08303_012E': 'commute_60_89min',    # commute 60-89 minutes
    'B08303_013E': 'commute_90plus_min',  # commute 90+ minutes
    'B25003_001E': 'occupied_units',      # occupied housing units
    'B25003_003E': 'renter_units',        # renter-occupied units
    'B25070_001E': 'renter_hh',           # renter households (rent burden base)
    'B25070_007E': 'rent_30_35pct',       # rent is 30-34.9% of income
    'B25070_008E': 'rent_35_40pct',       # rent is 35-39.9% of income
    'B25070_009E': 'rent_40_50pct',       # rent is 40-49.9% of income
    'B25070_010E': 'rent_50plus_pct',     # rent is 50%+ of income
    'B23025_003E': 'labor_force',         # civilian labor force
    'B23025_005E': 'unemployed',          # unemployed
    'state':       'state_fips',          # 25 = Massachusetts
    'county':      'county_fips',         # 025 = Suffolk County
    'tract':       'tract_code',          # 6-digit tract code
}

acs19 = acs19.rename(columns=rename_map)

Not renamed: []


In [ ]:
# GEOID = state + county + tract (11 digits)
acs19['GEOID'] = acs19['state_fips'] + acs19['county_fips'] + acs19['tract_code']
acs19 = acs19.drop(columns=['state_fips', 'county_fips'])

(204, 28)


In [ ]:
# Dropping Chelsea (16xx), Revere (17xx), Winthrop (18xx), non-residential (98xx, 99xx)
drop_prefixes = ['16', '17', '18', '98', '99']
acs19_boston = acs19[acs19['tract_code'].str[:2].isin(drop_prefixes) == False].reset_index(drop=True)

(167, 28)


In [5]:
a = acs19_boston
a['pct_poverty']        = a['pov_below'] / a['pov_universe']
a['pct_no_vehicle']     = a['hh_no_vehicle'] / a['households']
a['pct_minority']       = 1 - a['pop_nh_white'] / a['pop_total']
a['pct_black']          = a['pop_nh_black'] / a['pop_total']
a['pct_hispanic']       = a['pop_hispanic'] / a['pop_total']
a['pct_transit']        = a['workers_transit'] / a['workers']
a['pct_long_commute']   = (a['commute_45_59min'] + a['commute_60_89min'] + a['commute_90plus_min']) / a['commuters']
a['pct_renters']        = a['renter_units'] / a['occupied_units']
a['pct_rent_burdened']  = (a['rent_30_35pct'] + a['rent_35_40pct'] + a['rent_40_50pct'] + a['rent_50plus_pct']) / a['renter_hh']
a['unemployment_rate']  = a['unemployed'] / a['labor_force']

a.filter(like='pct_').describe().round(2)

,pct_poverty,pct_no_vehicle,pct_minority,pct_black,pct_hispanic,pct_transit,pct_long_commute,pct_renters,pct_rent_burdened
count,167.00,167.00,167.00,167.00,167.00,167.00,167.00,167.00,167.00
mean,0.19,0.35,0.54,0.21,0.20,0.33,0.23,0.66,0.47
std,0.12,0.18,0.29,0.24,0.15,0.12,0.10,0.20,0.12
min,0.02,0.03,0.03,0.00,0.01,0.09,0.05,0.10,0.11
25%,0.09,0.22,0.28,0.03,0.09,0.25,0.15,0.54,0.40
50%,0.17,0.34,0.50,0.09,0.15,0.32,0.23,0.66,0.48
75%,0.27,0.48,0.79,0.38,0.28,0.41,0.31,0.79,0.55
max,0.53,0.78,0.99,0.83,0.70,0.73,0.45,1.00,0.72


In [ ]:
acs19_boston.to_csv(clean_path_19, index=False)

Saved (167, 38)
